1. This cell loads all the necessary components for the evaluation. First, it loads the MultiEURLEX test set containing 5,000 legal documents with all 7,390 possible labels. Then it downloads the EuroVoc descriptor file from GitHub, which maps label IDs to their English text descriptions. Next, it extracts the label IDs from the dataset and converts them to readable text using the descriptor file. After that, it loads the **LaBSE embedding model**. Finally, it embeds all 7,390 label descriptions into 384-dimensional vectors.

In [1]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import requests
from tqdm import tqdm
import pandas as pd

# Set language here — change to 'fr', 'nl', or 'de'
LANGUAGE = 'de'
LANGUAGE_NAME = 'German'

dataset = load_dataset('coastalcph/multi_eurlex', LANGUAGE, split='test',
                       label_level='all_levels', trust_remote_code=True)

classlabel = dataset.features["labels"].feature
print(f"Number of labels: {len(classlabel.names)}")

url = "https://raw.githubusercontent.com/nlpaueb/multi-eurlex/master/data/eurovoc_descriptors.json"
eurovoc_concepts = requests.get(url).json()
label_ids = classlabel.names

# Use native language labels instead of English
label_descriptors = [eurovoc_concepts[label_id][LANGUAGE] for label_id in label_ids]

model = SentenceTransformer('sentence-transformers/LaBSE')
label_embeddings = model.encode(label_descriptors, show_progress_bar=True, batch_size=32)

Number of labels: 7390


Batches:   0%|          | 0/231 [00:00<?, ?it/s]

2. This cell defines three standard evaluation metrics used in information retrieval and multi-label classification. The precision at k function calculates what fraction of the top k retrieved labels are actually correct by dividing the number of correct labels in the top k by k. The recall at k function calculates what fraction of the true labels were found in the top k predictions by dividing the number of correct labels found by the total number of true labels. The NDCG at k function computes normalized discounted cumulative gain, which gives higher scores when correct labels appear earlier in the ranking, with labels at position 1 weighted more than those at position 10. NDCG also compares the actual ranking to an ideal ranking where all true labels would be ranked first. All three metrics return values between 0 and 1, where higher is better.

In [2]:
# Evaluation metrics functions
def precision_at_k(y_true, y_pred, k):
    """Precision@k: fraction of top-k predictions that are correct"""
    top_k = y_pred[:k]
    relevant = sum(1 for label in top_k if label in y_true)
    return relevant / k

def recall_at_k(y_true, y_pred, k):
    """Recall@k: fraction of true labels found in top-k"""
    if len(y_true) == 0:
        return 0.0
    top_k = y_pred[:k]
    relevant = sum(1 for label in top_k if label in y_true)
    return relevant / len(y_true)

def ndcg_at_k(y_true, y_pred, k):
    """NDCG@k: Normalized Discounted Cumulative Gain"""
    top_k = y_pred[:k]
    dcg = sum((1 if label in y_true else 0) / np.log2(i + 2) 
              for i, label in enumerate(top_k))
    
    # Ideal DCG (if all true labels were ranked first)
    ideal_k = min(len(y_true), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_k))
    
    return dcg / idcg if idcg > 0 else 0.0

print("Metric functions defined ✓")

Metric functions defined ✓


3. This cell processes every document in the 5,000 document test set to calculate retrieval performance. For each document, it first extracts the true labels that humans assigned to that document. Then it embeds the document text into a 384-dimensional vector using the same embedding model used for labels. Next, it calculates cosine similarity between the document vector and all 7,390 label vectors to measure how semantically similar each label is to the document. The labels are then ranked from highest to lowest similarity score. For each k value in the list of 5, 10, 20, and 50, the cell calculates precision, recall, and NDCG using the top k predicted labels compared to the true labels. All metric scores are stored in lists so that averages and standard deviations can be computed later. The entire process takes approximately 10 to 15 minutes to complete with a progress bar showing how many documents have been processed.

In [3]:
k_values = [5, 10, 20, 50, 100]
results = {
    'precision': {k: [] for k in k_values},
    'recall': {k: [] for k in k_values},
    'ndcg': {k: [] for k in k_values}
}

# Pre-encode all documents in batches
texts = [doc['text'] for doc in dataset]
doc_embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)

# Evaluate
for idx, doc in enumerate(tqdm(dataset, desc="Evaluating")):
    true_labels = set(doc['labels'])
    if len(true_labels) == 0:
        continue
    similarities = cosine_similarity([doc_embeddings[idx]], label_embeddings)[0]
    ranked_predictions = np.argsort(similarities)[::-1]
    for k in k_values:
        results['precision'][k].append(precision_at_k(true_labels, ranked_predictions, k))
        results['recall'][k].append(recall_at_k(true_labels, ranked_predictions, k))
        results['ndcg'][k].append(ndcg_at_k(true_labels, ranked_predictions, k))

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 5000/5000 [00:58<00:00, 85.39it/s] 


4. This cell takes all the metric scores collected during evaluation and computes summary statistics. For each metric and each k value, it calculates the mean score across all 5,000 documents and the standard deviation to show how much variation exists. The results are printed in a readable format showing precision, recall, and NDCG for k equals 5, 10, 20, and 50. Then it creates a pandas dataframe to display all results in a clean table format with columns for k, precision, recall, and NDCG. This table provides a quick overview of model performance at different retrieval depths, making it easy to see that recall increases with larger k values while precision typically decreases.

In [6]:
for metric_name, metric_data in results.items():
    for k in k_values:
        scores = metric_data[k]
        mean = np.mean(scores)
        std = np.std(scores)

summary_data = []
for k in k_values:
    summary_data.append({
        'k': k,
        'Precision': f"{np.mean(results['precision'][k]):.4f} ± {np.std(results['precision'][k]):.4f}",
        'Recall': f"{np.mean(results['recall'][k]):.4f} ± {np.std(results['recall'][k]):.4f}",
        'NDCG': f"{np.mean(results['ndcg'][k]):.4f} ± {np.std(results['ndcg'][k]):.4f}"
    })
summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

  k       Precision          Recall            NDCG
  5 0.0165 ± 0.0566 0.0140 ± 0.0510 0.0181 ± 0.0668
 10 0.0149 ± 0.0382 0.0255 ± 0.0687 0.0223 ± 0.0641
 20 0.0131 ± 0.0253 0.0444 ± 0.0905 0.0307 ± 0.0688
 50 0.0102 ± 0.0142 0.0849 ± 0.1239 0.0450 ± 0.0751
100 0.0077 ± 0.0088 0.1284 ± 0.1489 0.0577 ± 0.0787


5. This cell takes all the metric scores collected during evaluation and computes summary statistics. For each metric and each k value, it calculates the mean score across all 5,000 documents and the standard deviation to show how much variation exists. The results are printed in a readable format showing precision, recall, and NDCG for k equals 5, 10, 20, and 50. Then it creates a pandas dataframe to display all results in a clean table format with columns for k, precision, recall, and NDCG. This table provides a quick overview of model performance at different retrieval depths, making it easy to see that recall increases with larger k values while precision typically decreases.

In [7]:
import json

results_to_save = {
    'model': 'LaBSE',
    'dataset': 'MultiEURLEX',
    'language': f'{LANGUAGE_NAME} (native labels)',
    'num_documents': len(dataset),
    'num_labels': len(label_descriptors),
    'metrics': {
        metric_name: {
            k: {
                'mean': float(np.mean(scores)),
                'std': float(np.std(scores)),
                'values': [float(x) for x in scores]
            }
            for k, scores in metric_data.items()
        }
        for metric_name, metric_data in results.items()
    }
}

with open(f'results_labse_{LANGUAGE_NAME.lower()}_nativelabels.json', 'w') as f:
    json.dump(results_to_save, f, indent=2)